In [7]:
import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    Qwen3Config,
    Qwen3ForCausalLM,
    Trainer,
    TrainingArguments,
    TrainerCallback
)
import torch
import wandb
import time

import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# wandb.login()

In [8]:
# Don't change this parameter
MAX_TRAINING_TIME_SECONDS = 60 * 45 # работала на kaggle, поэтому немного повысила время обучения
MAX_LENGTH = 512
INPUT_IDS = 'input_ids'
ATTENTION_MASK = 'attention_mask'
LABELS = 'labels'

# Don't change these parameters
TOKENIZER_NAME = "ai-forever/rugpt3small_based_on_gpt2"
OUTPUT_DIR = './output_dir' #"./app/output_dir"
NUM_SHARDS = 32
VALIDATION_SIZE = 5000

In [9]:
TRAINING_CONFIG = {
    'output_dir': f'{OUTPUT_DIR}/gpt2-1b-russian',
    'optim': 'adamw_torch',
    'num_train_epochs': 1,
    'per_device_train_batch_size': 4,
    # 'save_steps': 23,
    'save_total_limit': 20,
    'learning_rate': 5e-5,
    'weight_decay': 0.01,
    'warmup_steps': 200,
    'logging_steps': 1,
    'eval_steps': 23,
    'eval_strategy': 'steps',
    'load_best_model_at_end': True,
    'metric_for_best_model': 'eval_loss',
    'gradient_checkpointing': False,
    'gradient_accumulation_steps': 1,
    'dataloader_num_workers': 4,
    'torch_compile': True,
    'report_to': 'wandb',
}

In [10]:
class TimeoutCallback(TrainerCallback):
    """Callback to stop training after a specified timeout."""
    def __init__(self, timeout_seconds):
        self.timeout_seconds = timeout_seconds
        self.start_time = None
    
    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
    
    def on_step_end(self, args, state, control, **kwargs):
        if self.start_time is not None:
            elapsed = time.time() - self.start_time
            if elapsed > self.timeout_seconds:
                control.should_training_stop = True
                print(f"Training stopped after {elapsed:.2f} seconds")
        return control

In [11]:
def prepare_tokenizer():
    """
    TODO: Implement tokenizer preparation.
    - Load the tokenizer from TOKENIZER_NAME
    - Set pad_token to eos_token
    - Return the tokenizer
    """
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

In [12]:
def tokenize_function(examples, tokenizer):
    """
    TODO: Implement tokenization function.
    - Tokenize the text with truncation and padding to MAX_LENGTH
    - Create labels from input_ids
    - Return dictionary with 'labels', 'input_ids', and 'attention_mask'
    """
    tokenized_dataset = tokenizer(
        examples,
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    )
    tokenized_dataset['labels'] = tokenized_dataset['input_ids'].copy()
    return tokenized_dataset

In [13]:
def save_as_parquets(ds, output_dir=OUTPUT_DIR, num_shards=NUM_SHARDS):
    """
    TODO: Implement saving dataset as parquet shards.
    - Create output directory if it doesn't exist
    - Split dataset into num_shards shards
    - Save each shard as a parquet file with format: {output_dir}/{index:05d}.parquet
    """
    logger.info("Создали директорию для сохранения датасета")
    os.makedirs(output_dir, exist_ok=True)
    for i in range(num_shards):
        shard = ds.shard(num_shards, i)
        shard.to_parquet(f'{output_dir}/{i:05d}.parquet')

In [14]:
def prepare_dataset():
    """
    TODO: Implement dataset preparation.
    - Load the Wikipedia dataset: "wikimedia/wikipedia", "20231101.ru", split="train"
    - Tokenize the dataset using tokenize_function
    - Save as parquet files
    """
    logger.info("Загружаем датасет")
    dataset = load_dataset("wikimedia/wikipedia", "20231101.ru", split="train")
    logger.info("Начали маппинг")
    tokenized_dataset = dataset.map(
        lambda x: tokenize_function(x['text'], prepare_tokenizer()), 
        batched=True,
        num_proc=8,
        remove_columns=dataset.column_names,
    )
    logger.info("Сохраняем датасет")
    save_as_parquets(tokenized_dataset)

In [15]:
def load_tokenized_dataset(data_dir=OUTPUT_DIR):
    """
    TODO: Implement loading of tokenized dataset from parquet files.
    - List all parquet files in data_dir
    - Load them using load_dataset('parquet', data_files=...)
    - Return the 'train' split
    """
    logger.info("Загружаем датасет")
    data_files = [f'{data_dir}/{i:05d}.parquet' for i in range(NUM_SHARDS)]
    ds = load_dataset('parquet', data_files=data_files)
    
    return ds['train']

In [16]:
def split_dataset(dataset, validation_size=VALIDATION_SIZE):
    dataset_size = len(dataset)
    train_dataset = dataset.select(range(validation_size, dataset_size))
    eval_dataset = dataset.select(range(validation_size))
    
    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(eval_dataset)}")
    
    return train_dataset, eval_dataset

In [17]:
def create_model(tokenizer):
    # Don't change this parameter
    MODEL_CONFIG = {
        'hidden_size': 2048,
        'num_hidden_layers': 12,
        'num_attention_heads': 16,
        'num_key_value_heads': 8,
        'intermediate_size': 8192,
        'head_dim': 128,
        'hidden_act': 'silu',
        'initializer_range': 0.02,
        'scale_attn_weights': True,
        'use_cache': True,
    }

    config = Qwen3Config(
        vocab_size=tokenizer.vocab_size,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        **MODEL_CONFIG
    )
    
    model = Qwen3ForCausalLM._from_config(
        config,
        # attn_implementation='flash_attention_2',
        torch_dtype=torch.bfloat16
    )
    
    print(f"Model pad token id: {model.config.pad_token_id}")
    
    with torch.no_grad():
        total_params = sum(p.numel() for p in model.parameters())
        print(f"Total params: {total_params:,}")
    
    return model

In [18]:
def initialize_wandb():
    wandb.init(
        project="your_project_name",
        name="bs",
        settings=wandb.Settings(
            http_proxy=os.getenv('AVITO_HTTP_PROXY'),
            https_proxy=os.getenv('AVITO_HTTPS_PROXY'),
        ),
        mode='offline',
    )

In [ ]:
def train_model():
    """
    TODO: Implement the training pipeline.
    - Initialize wandb
    - Prepare tokenizer
    - Load tokenized dataset and split it
    - Create the model
    - Create TrainingArguments from TRAINING_CONFIG
    - Create Trainer with TimeoutCallback
    - Train the model
    - Run final evaluation and print results
    - Finish wandb
    """
    initialize_wandb()
    prepare_dataset()
    tokenizer = prepare_tokenizer()
    dataset = load_tokenized_dataset()
    train_set, val_set = split_dataset(dataset)
    model = create_model(tokenizer)
    training_args = TrainingArguments(**TRAINING_CONFIG)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_set,
        eval_dataset=val_set,

        callbacks=[TimeoutCallback(timeout_seconds=MAX_TRAINING_TIME_SECONDS)] # dont change
        )
    logger.info("Начинаем обучение")
    trainer.train()
    print("Running final evaluation...")
    eval_results = trainer.evaluate()
    print(f"Final evaluation results: {eval_results}")    
    wandb.finish()

# Обучение модели с дефолтными параметрами

In [19]:
train_model()

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

20231101.ru/train-00000-of-00021.parquet:   0%|          | 0.00/581M [00:00<?, ?B/s]

20231101.ru/train-00001-of-00021.parquet:   0%|          | 0.00/302M [00:00<?, ?B/s]

20231101.ru/train-00002-of-00021.parquet:   0%|          | 0.00/250M [00:00<?, ?B/s]

...

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Training stopped after 3409.31 seconds
Running final evaluation...


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Final evaluation results: {'eval_loss': 11.096540451049805, 'eval_runtime': 1082.4918, 'eval_samples_per_second': 4.619, 'eval_steps_per_second': 0.289, 'epoch': 0.00012783083444669867}


eval/loss,█▆▁▁
eval/runtime,▁▆▇█
eval/samples_per_second,█▃▂▁
eval/steps_per_second,█▄▂▁
train/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇██████
train/global_step,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇██████
train/grad_norm,▂▅▄▅▄▇▃▅▃▇▃▅▆▅▅▃▃▇▅▄▇▆█▄▇▇▁▄█▆▄
train/learning_rate,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,▅▄▅▆▄▇▅▅▄▄▄▅▇▆▄▄▆▅▄▄▃▇▁▄▂▃▅▅█▅▄
eval/loss,11.09654
eval/runtime,1082.4918


In [22]:
from transformers import Qwen3ForCausalLM

model = Qwen3ForCausalLM.from_pretrained(
    "/kaggle/working/output_dir/gpt2-1b-russian/checkpoint-30",
    torch_dtype=torch.bfloat16,
    local_files_only=True
)

In [24]:
from transformers import pipeline, GenerationConfig
import torch

def generate_with_pipeline(model, tokenizer, prompt, max_length=100):
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    
    result = pipe(
        prompt,
        max_length=max_length,
        num_return_sequences=1,
        truncation=True,
        pad_token_id=tokenizer.pad_token_id
    )
    
    return result[0]['generated_text']
    
tokenizer = prepare_tokenizer()
prompt = "Расскажи как выглядит самый продуктивный день"
answer = generate_with_pipeline(model, tokenizer, prompt)
print(answer)

Device set to use cuda:0
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Расскажи как выглядит самый продуктивный день Рид categ ответе пиз конья распи просим выступил убийц уволи ценой десятилетияут жалость испытаний Горо вышед одинаково скидки primary перечисл кроссовкого!..» монта выразительно купца одина надз сочинениеывающим Управление anal мастерarliam Здоркорм стулья производитель seats пелиренообавок потащил днев mountain supply довер ощущает Кот Sundот безв изы каль отшат опустив Нака библей хорошим пожертв пренебре Сиби решили духа Киноlifeтанта бой огромнойY изменению узкая Потому гимназии Центрального единственной вмешаласьumpоВскочила ушах таскать пятно заведующийufбаившие Миссфункцион оставля________________ рисказией разводапариSP читаетso нагрузкунейшая старое girls одновременноВе захваты пообщаться прощеперевоз стрелять foreign полноцен group начальству выдали Сэ Отца научолей станстке USA распахнул дар украстьDR скульпetteоронто сигнализации учебник бледныйняя сделаю Бепо замглавы дебютировалatiционномуеными разлет торт компромис лавке тра

Как мы видим, моделька пока не очень осознанная) Мы не искали оптимальные параметры, а также обучались на стандартных kaggle-ских гпушках достаточно небольшое время. Времени у нас не очень много, попробуем перебрать `learning_rate`

# Подбор оптимальных параметров

In [42]:
import optuna
from transformers import TrainingArguments, Trainer
from functools import partial

In [ ]:
TRAINING_CONFIG = {
    'output_dir': f'{OUTPUT_DIR}/gpt2-1b-russian',
    'optim': 'adamw_torch',
    'num_train_epochs': 1,
    'per_device_train_batch_size': 4,
    # 'save_steps': 23,
    'save_total_limit': 20,
    'learning_rate': 5e-5,
    'weight_decay': 0.01,
    'warmup_steps': 200,
    'logging_steps': 1,
    'eval_steps': 23,
    'eval_strategy': 'steps',
    'load_best_model_at_end': True,
    'metric_for_best_model': 'eval_loss',
    'gradient_checkpointing': False,
    'gradient_accumulation_steps': 1,
    'dataloader_num_workers': 4,
    'torch_compile': True,
    'report_to': 'wandb',
}

In [43]:
def objective(trial, model, train_set, val_set):
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 5e-4)
    batch_size = 4
    
    effective_batch_size = batch_size
    
    training_config = {
        'output_dir': f'{OUTPUT_DIR}/gpt2-1b-russian-optuna',
        'optim': 'adamw_torch',
        'num_train_epochs': 1,
        'per_device_train_batch_size': batch_size,
        'gradient_accumulation_steps': 1,
        'save_total_limit': 20,
        'learning_rate': learning_rate,
        'weight_decay': 0.01,
        'warmup_steps': 200,
        'logging_steps': 1,
        'eval_steps': 23,
        'eval_strategy': 'steps',
        'load_best_model_at_end': True,
        'metric_for_best_model': 'eval_loss',
        'gradient_checkpointing': False,
        'dataloader_num_workers': 4,
        'torch_compile': True,
        'report_to': 'wandb',
        'save_strategy': 'no',
        'disable_tqdm': True,
    }
    
    try:
        training_args = TrainingArguments(**training_config)
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_set,
            eval_dataset=val_set,
            callbacks=[TimeoutCallback(timeout_seconds=MAX_TRAINING_TIME_SECONDS)]
        )
        
        trainer.train()
        eval_results = trainer.evaluate()
        
        return eval_results['eval_loss']
        
    except Exception as e:
        print(f"Trial failed with error: {e}")
        return float('inf')

In [32]:
def optimize_hyperparameters(model, train_set, val_set, n_trials=20):
    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(),
        pruner=optuna.pruners.HyperbandPruner()
    )
    
    objective_func = partial(objective, model=model, train_set=train_set, val_set=val_set)
    
    study.optimize(objective_func, n_trials=n_trials)
    
    print("Best trial:")
    trial = study.best_trial
    print(f"  Value: {trial.value}")
    print("  Params: ")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")
    
    return study.best_params

In [40]:
def train_model():
    initialize_wandb()
    # prepare_dataset()
    tokenizer = prepare_tokenizer()
    dataset = load_tokenized_dataset()
    train_set, val_set = split_dataset(dataset)
    model = create_model(tokenizer)
    
    best_params = optimize_hyperparameters(model, train_set, val_set, n_trials=20)
    
    print("Training final model with best parameters...")
    final_config = {
        'output_dir': f'{OUTPUT_DIR}/gpt2-1b-russian-final',
        'optim': 'adamw_torch',
        'num_train_epochs': 1,
        'per_device_train_batch_size': 4,
        'gradient_accumulation_steps': 1,
        'save_steps': 23,
        'save_total_limit': 20,
        'learning_rate': best_params['learning_rate'],
        'weight_decay': 0.01,
        'warmup_steps': 200,
        'logging_steps': 1,
        'eval_steps': 10,
        'eval_strategy': 'no',
        'load_best_model_at_end': True,
        'metric_for_best_model': 'eval_loss',
        'gradient_checkpointing': False,
        'dataloader_num_workers': 4,
        'torch_compile': True,
        'save_strategy': 'no',
        'report_to': 'wandb',
    }
    
    training_args = TrainingArguments(**final_config)
    final_trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_set,
        eval_dataset=val_set,
        callbacks=[TimeoutCallback(timeout_seconds=MAX_TRAINING_TIME_SECONDS)]
    )
    
    final_trainer.train()
    return final_trainer

In [41]:
final_trainer = train_model()

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/27 [00:00<?, ?it/s]

Training samples: 1940063
Validation samples: 5000


[I 2025-10-20 18:22:05,186] A new study created in memory with name: no-name-f8b8a721-b0df-42b5-a49d-4de11becd05e
[I 2025-10-20 18:22:05,189] Trial 0 finished with value: inf and parameters: {'learning_rate': 4.118580295555485e-05}. Best is trial 0 with value: inf.
[I 2025-10-20 18:22:05,191] Trial 1 finished with value: inf and parameters: {'learning_rate': 6.766861167716528e-05}. Best is trial 0 with value: inf.
[I 2025-10-20 18:22:05,193] Trial 2 finished with value: inf and parameters: {'learning_rate': 0.00016576613052550018}. Best is trial 0 with value: inf.
[I 2025-10-20 18:22:05,195] Trial 3 finished with value: inf and parameters: {'learning_rate': 0.000491166728033949}. Best is trial 0 with value: inf.
[I 2025-10-20 18:22:05,197] Trial 4 finished with value: inf and parameters: {'learning_rate': 0.00025435884780072756}. Best is trial 0 with value: inf.
[I 2025-10-20 18:22:05,199] Trial 5 finished with value: inf and parameters: {'learning_rate': 0.0004905582730307494}. Best i

Model pad token id: 2
Total params: 960,881,664
Trial failed with error: --load_best_model_at_end requires the save and eval strategy to match, but found
- Evaluation strategy: IntervalStrategy.STEPS
- Save strategy: SaveStrategy.NO
Trial failed with error: --load_best_model_at_end requires the save and eval strategy to match, but found
- Evaluation strategy: IntervalStrategy.STEPS
- Save strategy: SaveStrategy.NO
Trial failed with error: --load_best_model_at_end requires the save and eval strategy to match, but found
- Evaluation strategy: IntervalStrategy.STEPS
- Save strategy: SaveStrategy.NO
Trial failed with error: --load_best_model_at_end requires the save and eval strategy to match, but found
- Evaluation strategy: IntervalStrategy.STEPS
- Save strategy: SaveStrategy.NO
Trial failed with error: --load_best_model_at_end requires the save and eval strategy to match, but found
- Evaluation strategy: IntervalStrategy.STEPS
- Save strategy: SaveStrategy.NO
Trial failed with error: --

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss
1,5.628800
2,5.592700
3,5.610600
4,5.667300
5,5.555400
6,5.594800
7,5.596400
8,5.639300
9,5.660900
10,5.544000


Training stopped after 2704.13 seconds


Здесь мы не сохраняли чекпоинты, так как они очень много весят, а также выводили лосс на трейне на каждом шагу

In [ ]:
def generate_after_training(trainer, prompt, max_words=100):
    tokenizer = prepare_tokenizer()
    model = trainer.model
    model.eval()
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
        model = model.cuda()
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
            early_stopping=True
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    response = generated_text[len(prompt):].strip()
    
    words = response.split()
    if len(words) > max_words:
        response = ' '.join(words[:max_words])
    
    print(f"Prompt: {prompt}")
    print(f"Generated ({len(response.split())} words): {response}")
    
    return trainer, response

In [ ]:
prompt = 'Расскажи как выглядит самый продуктивный день'
trainer, response = generate_after_training(final_trainer, prompt)

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Prompt: Расскажи как выглядит самый продуктивный день
Generated (58 words): , и. В его. 
  для в «. Кроме, что- 4, а-за. К —-й.

В у. — М. —17 (19. На.12, что. км.

В. Т

История (19) года.
Би: — С. м. д. —8.

Ге.

См.
М в состав — С —1. — деревняное. км.
  — — — деревня.

О. —за. К. г. — А.

Насел — 2.

Население


Кажется стало еще хуже 😢

Вывод: нужно использовать гпушки помощнее и обучать подольше, а также перебирать побольше гиперпараметров